# Predictions02 - Using v2 Models (Enhanced Features)
Loads models trained in **Notebook03** with 24 features:
- H2H last 5 matches
- Recent form (last 5) — avg scored/conceded only
- Home vs Away win rate split
- Competition importance (1-4)
- FIFA ranking & rank difference

In [1]:
# ============================================================
# CELL 1 - Load Data & Normalize Country Names
# ============================================================
import pandas as pd
import numpy as np
import joblib

results      = pd.read_csv('../data/results.csv')
shootouts    = pd.read_csv('../data/shootouts.csv')
former_names = pd.read_csv('../data/former_names.csv')
rankings     = pd.read_csv('../dataII/rankings.csv')

results['date']   = pd.to_datetime(results['date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])

name_map = dict(zip(former_names['former'], former_names['current']))
def normalize(name): return name_map.get(name, name)

results['home_team']   = results['home_team'].apply(normalize)
results['away_team']   = results['away_team'].apply(normalize)
shootouts['home_team'] = shootouts['home_team'].apply(normalize)
shootouts['away_team'] = shootouts['away_team'].apply(normalize)
shootouts['winner']    = shootouts['winner'].apply(normalize)

results_sorted = results.dropna(subset=['home_score','away_score']).sort_values('date').reset_index(drop=True)

rank_lookup   = dict(zip(rankings['team'], rankings['fifa_rank']))
points_lookup = dict(zip(rankings['team'], rankings['fifa_points']))

print('Data loaded!')
print('results:', results_sorted.shape, '  rankings:', rankings.shape)

Data loaded!
results: (49448, 9)   rankings: (216, 4)


In [2]:
# ============================================================
# CELL 2 - Helper Functions (same as Notebook03)
# ============================================================

def get_rank(team):
    return rank_lookup.get(team, 200)

def team_stats(team, before_date, n=20):
    past = results_sorted[
        (results_sorted['date'] < before_date) &
        ((results_sorted['home_team'] == team) | (results_sorted['away_team'] == team))
    ].tail(n)
    if len(past) == 0:
        return {'win_rate': 0.5, 'avg_scored': 1.0, 'avg_conceded': 1.0, 'n_matches': 0}
    wins, scored, conceded = [], [], []
    for _, m in past.iterrows():
        if m['home_team'] == team:
            wins.append(1 if m['home_score'] > m['away_score'] else 0)
            scored.append(m['home_score']); conceded.append(m['away_score'])
        else:
            wins.append(1 if m['away_score'] > m['home_score'] else 0)
            scored.append(m['away_score']); conceded.append(m['home_score'])
    return {'win_rate': np.mean(wins), 'avg_scored': np.mean(scored),
            'avg_conceded': np.mean(conceded), 'n_matches': len(past)}

def team_stats_split(team, before_date, n=20):
    past_home = results_sorted[
        (results_sorted['date'] < before_date) & (results_sorted['home_team'] == team)
    ].tail(n)
    past_away = results_sorted[
        (results_sorted['date'] < before_date) & (results_sorted['away_team'] == team)
    ].tail(n)
    home_wr = (past_home['home_score'] > past_home['away_score']).mean() if len(past_home) > 0 else 0.5
    away_wr = (past_away['away_score'] > past_away['home_score']).mean() if len(past_away) > 0 else 0.5
    return float(home_wr), float(away_wr)

def recent_form(team, before_date, n=5):
    s = team_stats(team, before_date, n=n)
    return s['win_rate'], s['avg_scored'], s['avg_conceded']

def h2h_stats(home_team, away_team, before_date, n=5):
    past = results_sorted[
        (results_sorted['date'] < before_date) &
        (
            ((results_sorted['home_team'] == home_team) & (results_sorted['away_team'] == away_team)) |
            ((results_sorted['home_team'] == away_team) & (results_sorted['away_team'] == home_team))
        )
    ].tail(n)
    if len(past) == 0:
        return {'h2h_home_win_rate': 0.5, 'h2h_avg_home_scored': 1.0, 'h2h_avg_away_scored': 1.0, 'h2h_n': 0}
    home_wins, home_scored, away_scored = [], [], []
    for _, m in past.iterrows():
        if m['home_team'] == home_team:
            home_wins.append(1 if m['home_score'] > m['away_score'] else 0)
            home_scored.append(m['home_score']); away_scored.append(m['away_score'])
        else:
            home_wins.append(1 if m['away_score'] > m['home_score'] else 0)
            home_scored.append(m['away_score']); away_scored.append(m['home_score'])
    return {
        'h2h_home_win_rate':   np.mean(home_wins),
        'h2h_avg_home_scored': np.mean(home_scored),
        'h2h_avg_away_scored': np.mean(away_scored),
        'h2h_n':               len(past)
    }

def comp_importance(tournament):
    t = str(tournament).lower()
    if 'fifa world cup' in t:                                              return 4
    if any(x in t for x in ['euro', 'copa america', 'asian cup',
                              'africa cup', 'gold cup', 'nations cup']):   return 3
    if any(x in t for x in ['qualifier', 'qualification']):               return 2
    return 1

def shootout_win_rate(team, before_date):
    past = shootouts[
        (shootouts['date'] < before_date) &
        ((shootouts['home_team'] == team) | (shootouts['away_team'] == team))
    ]
    if len(past) == 0: return 0.5
    return (past['winner'] == team).sum() / len(past)

print('Helper functions ready!')

Helper functions ready!


In [3]:
# ============================================================
# CELL 3 - Load v2 Models
# ============================================================
model_outcome    = joblib.load('../dataII/model_outcome_v2.pkl')
model_home_score = joblib.load('../dataII/model_home_score_v2.pkl')
model_away_score = joblib.load('../dataII/model_away_score_v2.pkl')

FEATURES = [
    'is_neutral', 'year', 'comp_importance',
    'home_win_rate', 'home_avg_scored', 'home_avg_conceded',
    'home_home_win_rate', 'home_away_win_rate',
    'home_form_avg_scored', 'home_form_avg_conceded',
    'away_win_rate', 'away_avg_scored', 'away_avg_conceded',
    'away_home_win_rate', 'away_away_win_rate',
    'away_form_avg_scored', 'away_form_avg_conceded',
    'h2h_home_win_rate', 'h2h_avg_home_scored', 'h2h_avg_away_scored', 'h2h_n',
    'home_fifa_rank', 'away_fifa_rank', 'rank_diff',
]

print(f'Loaded: {type(model_outcome).__name__} (outcome)')
print(f'Loaded: {type(model_home_score).__name__} (home score)')
print(f'Loaded: {type(model_away_score).__name__} (away score)')
print(f'Feature count: {len(FEATURES)}')

Loaded: GradientBoostingClassifier (outcome)
Loaded: Ridge (home score)
Loaded: Ridge (away score)
Feature count: 24


In [4]:
# ============================================================
# CELL 4 - Predict Function (v2)
# ============================================================
def predict(home_team, away_team, is_neutral=0, tournament='Friendly'):
    today = pd.Timestamp.today()

    h           = team_stats(home_team, today)
    a           = team_stats(away_team, today)
    h_hw, h_aw  = team_stats_split(home_team, today)
    a_hw, a_aw  = team_stats_split(away_team, today)
    _, h_fsc, h_fcc = recent_form(home_team, today)
    _, a_fsc, a_fcc = recent_form(away_team, today)
    h2h         = h2h_stats(home_team, away_team, today)
    home_rank   = get_rank(home_team)
    away_rank   = get_rank(away_team)

    row = pd.DataFrame([{
        'is_neutral':             is_neutral,
        'year':                   today.year,
        'comp_importance':        comp_importance(tournament),
        'home_win_rate':          h['win_rate'],
        'home_avg_scored':        h['avg_scored'],
        'home_avg_conceded':      h['avg_conceded'],
        'home_home_win_rate':     h_hw,
        'home_away_win_rate':     h_aw,
        'home_form_avg_scored':   h_fsc,
        'home_form_avg_conceded': h_fcc,
        'away_win_rate':          a['win_rate'],
        'away_avg_scored':        a['avg_scored'],
        'away_avg_conceded':      a['avg_conceded'],
        'away_home_win_rate':     a_hw,
        'away_away_win_rate':     a_aw,
        'away_form_avg_scored':   a_fsc,
        'away_form_avg_conceded': a_fcc,
        'h2h_home_win_rate':      h2h['h2h_home_win_rate'],
        'h2h_avg_home_scored':    h2h['h2h_avg_home_scored'],
        'h2h_avg_away_scored':    h2h['h2h_avg_away_scored'],
        'h2h_n':                  h2h['h2h_n'],
        'home_fifa_rank':         home_rank,
        'away_fifa_rank':         away_rank,
        'rank_diff':              away_rank - home_rank,
    }])

    outcome = model_outcome.predict(row)[0]
    proba   = model_outcome.predict_proba(row)[0]
    home_g  = max(0, int(round(model_home_score.predict(row)[0])))
    away_g  = max(0, int(round(model_away_score.predict(row)[0])))
    label   = {0: 'Home Win', 1: 'Draw', 2: 'Away Win'}

    venue = 'Neutral' if is_neutral else f'{home_team} home ground'
    print(f'  {home_team}  vs  {away_team}')
    # print(f'  Tournament     : {tournament}  (importance={comp_importance(tournament)})')
    # print(f'  Venue          : {venue}')
    # print(f'  FIFA Ranks     : {home_team} #{int(home_rank)}  vs  {away_team} #{int(away_rank)}')
    print(f'  Predicted Score: {home_team} {home_g} - {away_g} {away_team}')
    print(f'  Outcome        : {label[outcome]}')
    print(f'  {home_team} win  : {proba[0]*100:.1f}%')
    print(f'  Draw           : {proba[1]*100:.1f}%')
    print(f'  {away_team} win  : {proba[2]*100:.1f}%')
    print()

print('predict() ready!')

predict() ready!


In [5]:
# ============================================================
# CELL 5 - Run Predictions
# ============================================================
# predict('Scotland',    'Morocco',  is_neutral=1, tournament='FIFA World Cup')
# predict('France',    'England',    is_neutral=1, tournament='FIFA World Cup')
# predict('Spain',     'Cape Verde',    is_neutral=1, tournament='FIFA World Cup')
# predict('Norway',    'Senegal',    is_neutral=1, tournament='FIFA World Cup')
# predict('Argentina', 'Iran',       is_neutral=1, tournament='FIFA World Cup')
# predict('Jordan',    'Algeria',    is_neutral=1, tournament='FIFA World Cup')
predict('Portugal',  'Uzbekistan', is_neutral=1, tournament='FIFA World Cup')
predict('England',   'Ghana',      is_neutral=1, tournament='FIFA World Cup')

  Portugal  vs  Uzbekistan
  Predicted Score: Portugal 2 - 1 Uzbekistan
  Outcome        : Home Win
  Portugal win  : 73.5%
  Draw           : 16.5%
  Uzbekistan win  : 10.0%

  England  vs  Ghana
  Predicted Score: England 2 - 1 Ghana
  Outcome        : Home Win
  England win  : 79.4%
  Draw           : 13.5%
  Ghana win  : 7.1%



In [6]:
predict('Panama', 'Croatia', is_neutral=1, tournament='FIFA World Cup')
predict('Colombia', 'DR Congo', is_neutral=1, tournament='FIFA World Cup')

  Panama  vs  Croatia
  Predicted Score: Panama 2 - 2 Croatia
  Outcome        : Away Win
  Panama win  : 18.8%
  Draw           : 27.8%
  Croatia win  : 53.4%

  Colombia  vs  DR Congo
  Predicted Score: Colombia 2 - 1 DR Congo
  Outcome        : Home Win
  Colombia win  : 48.5%
  Draw           : 31.7%
  DR Congo win  : 19.9%



In [7]:
predict('Canada', 'Switzerland', is_neutral=0, tournament='FIFA World Cup')
predict('Bosnia and Herzegovina', 'Qatar', is_neutral=1, tournament='FIFA World Cup')

  Canada  vs  Switzerland
  Predicted Score: Canada 2 - 1 Switzerland
  Outcome        : Home Win
  Canada win  : 35.7%
  Draw           : 34.7%
  Switzerland win  : 29.6%

  Bosnia and Herzegovina  vs  Qatar
  Predicted Score: Bosnia and Herzegovina 1 - 2 Qatar
  Outcome        : Away Win
  Bosnia and Herzegovina win  : 22.9%
  Draw           : 26.8%
  Qatar win  : 50.4%



In [8]:
predict('Morocco', 'Haiti', is_neutral=1, tournament='FIFA World Cup')
predict('Scotland', 'Brazil', is_neutral=1, tournament='FIFA World Cup')

  Morocco  vs  Haiti
  Predicted Score: Morocco 2 - 1 Haiti
  Outcome        : Home Win
  Morocco win  : 73.0%
  Draw           : 18.6%
  Haiti win  : 8.4%

  Scotland  vs  Brazil
  Predicted Score: Scotland 1 - 2 Brazil
  Outcome        : Away Win
  Scotland win  : 17.9%
  Draw           : 29.1%
  Brazil win  : 53.0%



In [9]:
predict('Czech Republic', 'Mexico', is_neutral=1, tournament='FIFA World Cup')
predict('South Africa', 'South Korea', is_neutral=1, tournament='FIFA World Cup')

  Czech Republic  vs  Mexico
  Predicted Score: Czech Republic 1 - 2 Mexico
  Outcome        : Away Win
  Czech Republic win  : 27.0%
  Draw           : 23.6%
  Mexico win  : 49.4%

  South Africa  vs  South Korea
  Predicted Score: South Africa 2 - 1 South Korea
  Outcome        : Draw
  South Africa win  : 34.1%
  Draw           : 34.4%
  South Korea win  : 31.5%



In [10]:
predict('Curaçao', 'Ivory Coast', is_neutral=1, tournament='FIFA World Cup')
predict('Ecuador', 'Germany', is_neutral=1, tournament='FIFA World Cup')

  Curaçao  vs  Ivory Coast
  Predicted Score: Curaçao 1 - 2 Ivory Coast
  Outcome        : Away Win
  Curaçao win  : 22.5%
  Draw           : 19.2%
  Ivory Coast win  : 58.3%

  Ecuador  vs  Germany
  Predicted Score: Ecuador 1 - 2 Germany
  Outcome        : Away Win
  Ecuador win  : 14.7%
  Draw           : 32.9%
  Germany win  : 52.4%



In [11]:
predict('Tunisia', 'Netherlands', is_neutral=1, tournament='FIFA World Cup')
predict('Japan', 'Sweden', is_neutral=1, tournament='FIFA World Cup')

  Tunisia  vs  Netherlands
  Predicted Score: Tunisia 1 - 2 Netherlands
  Outcome        : Away Win
  Tunisia win  : 12.1%
  Draw           : 17.9%
  Netherlands win  : 70.0%

  Japan  vs  Sweden
  Predicted Score: Japan 2 - 1 Sweden
  Outcome        : Home Win
  Japan win  : 68.5%
  Draw           : 18.8%
  Sweden win  : 12.7%



In [12]:
predict('Turkey', 'United States', is_neutral=0, tournament='FIFA World Cup')
predict('Paraguay', 'Australia', is_neutral=1, tournament='FIFA World Cup')

  Turkey  vs  United States
  Predicted Score: Turkey 3 - 1 United States
  Outcome        : Home Win
  Turkey win  : 67.4%
  Draw           : 18.3%
  United States win  : 14.3%

  Paraguay  vs  Australia
  Predicted Score: Paraguay 1 - 1 Australia
  Outcome        : Away Win
  Paraguay win  : 20.2%
  Draw           : 29.8%
  Australia win  : 50.0%



In [13]:
predict('Senegal', 'Iraq', is_neutral=1, tournament='FIFA World Cup')
predict('Norway', 'France', is_neutral=1, tournament='FIFA World Cup')

  Senegal  vs  Iraq
  Predicted Score: Senegal 2 - 1 Iraq
  Outcome        : Home Win
  Senegal win  : 56.7%
  Draw           : 28.8%
  Iraq win  : 14.5%

  Norway  vs  France
  Predicted Score: Norway 1 - 2 France
  Outcome        : Away Win
  Norway win  : 23.9%
  Draw           : 28.1%
  France win  : 48.1%



In [14]:
predict('Cape Verde', 'Saudi Arabia', is_neutral=1, tournament='FIFA World Cup')
predict('Uruguay', 'Spain', is_neutral=1, tournament='FIFA World Cup')

  Cape Verde  vs  Saudi Arabia
  Predicted Score: Cape Verde 1 - 2 Saudi Arabia
  Outcome        : Away Win
  Cape Verde win  : 34.8%
  Draw           : 24.6%
  Saudi Arabia win  : 40.6%

  Uruguay  vs  Spain
  Predicted Score: Uruguay 1 - 2 Spain
  Outcome        : Away Win
  Uruguay win  : 15.2%
  Draw           : 21.6%
  Spain win  : 63.2%



In [15]:
predict('Egypt', 'Iran', is_neutral=1, tournament='FIFA World Cup')
predict('New Zealand', 'Belgium', is_neutral=1, tournament='FIFA World Cup')

  Egypt  vs  Iran
  Predicted Score: Egypt 2 - 1 Iran
  Outcome        : Home Win
  Egypt win  : 33.9%
  Draw           : 32.5%
  Iran win  : 33.6%

  New Zealand  vs  Belgium
  Predicted Score: New Zealand 1 - 2 Belgium
  Outcome        : Away Win
  New Zealand win  : 4.5%
  Draw           : 15.1%
  Belgium win  : 80.4%



In [17]:
predict('Colombia', 'Portugal', is_neutral=1, tournament='FIFA World Cup')
predict('DR Congo', 'Uzbekistan', is_neutral=1, tournament='FIFA World Cup')

  Colombia  vs  Portugal
  Predicted Score: Colombia 2 - 2 Portugal
  Outcome        : Away Win
  Colombia win  : 26.4%
  Draw           : 28.4%
  Portugal win  : 45.2%

  DR Congo  vs  Uzbekistan
  Predicted Score: DR Congo 1 - 2 Uzbekistan
  Outcome        : Home Win
  DR Congo win  : 48.3%
  Draw           : 24.1%
  Uzbekistan win  : 27.6%



In [18]:
predict('Algeria', 'Austria', is_neutral=1, tournament='FIFA World Cup')
predict('Jordan', 'Argentina', is_neutral=1, tournament='FIFA World Cup')

  Algeria  vs  Austria
  Predicted Score: Algeria 1 - 2 Austria
  Outcome        : Draw
  Algeria win  : 30.9%
  Draw           : 35.2%
  Austria win  : 33.9%

  Jordan  vs  Argentina
  Predicted Score: Jordan 0 - 3 Argentina
  Outcome        : Away Win
  Jordan win  : 3.5%
  Draw           : 10.8%
  Argentina win  : 85.7%

